In [40]:
import pandas as pd
import csv
import joblib
import xgboost as xgb
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_fscore_support,
    matthews_corrcoef
)
import numpy as np
import tensorflow as tf
from tensorflow import keras
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from pyswarm import pso  # Librairie PSO



In [41]:
import pandas as pd

# Charger un fichier CSV
df = pd.read_csv('../datasets/Bias_correction_ucl.csv')

# Afficher les premières lignes du dataset


In [42]:
df.replace("-", np.nan, inplace=True)
print("Nombre total de valeurs nulles :", df.isnull().sum().sum())

Nombre total de valeurs nulles : 1248


In [43]:
import pandas as pd
import numpy as np

# Remplacer les tirets par des valeurs nulles
df.replace("-", np.nan, inplace=True)

# Remplacer les valeurs nulles par la valeur la plus fréquente de chaque colonne
for col in df.columns:
    mode_val = df[col].mode()[0]  # Obtenir la valeur la plus fréquente (mode) pour la colonne
    df[col] = df[col].fillna(mode_val)  # Remplacer les valeurs nulles par la valeur la plus fréquente

In [44]:
# 1. Extraction de l'année, mois, jour, heure, minute, seconde de 'Timestamp'
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day

In [45]:
df = df.drop(columns=df.select_dtypes(include=['datetime']).columns)

In [46]:
df.to_csv('df_cleaned.csv', index=False)

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Définir les caractéristiques (X) et les cibles (y)
X = df.drop(columns=["Next_Tmin", "Next_Tmax"])  # Exemple de caractéristiques d'entrée
y = df[['Next_Tmin']]  # Variables cibles : température minimale et maximale futures

# Séparer les données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Appliquer le MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Linear Regression

In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
# Créer et entraîner le modèle de régression linéaire
model = LinearRegression()
model.fit(X_train_scaled, y_train)
# Faire des prédictions
y_pred = model.predict(X_test_scaled)

# MAE : Erreur absolue moyenne
mae = mean_absolute_error(y_test, y_pred)

# MSE : Erreur quadratique moyenne
mse = mean_squared_error(y_test, y_pred)

# RMSE : Racine de l’erreur quadratique moyenne
rmse = np.sqrt(mse)

# R² : Coefficient de détermination
r2 = r2_score(y_test, y_pred)

# Affichage
print("📏 MAE  :", mae)
print("📏 MSE  :", mse)
print("📏 RMSE :", rmse)
print("📈 R²   :", r2)


📏 MAE  : 0.8312903758855885
📏 MSE  : 1.2340677563058449
📏 RMSE : 1.1108860230941089
📈 R²   : 0.8019947081880094


In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# Création du modèle multi-sortie
rf = RandomForestRegressor(n_estimators=100, random_state=42)
multi_rf = MultiOutputRegressor(rf)

# Entraînement
multi_rf.fit(X_train_scaled, y_train)

# Prédictions
y_pred = multi_rf.predict(X_test_scaled)

# Évaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)


📏 MAE : 0.562928433268859
📏 MSE : 0.5628011966473249
📏 RMSE : 0.7502007708922491
📈 R²  : 0.9096989491825983


In [12]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Création du modèle XGBoost pour multi-sortie
xgb = XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, random_state=42)
multi_xgb = MultiOutputRegressor(xgb)

# Entraînement
multi_xgb.fit(X_train_scaled, y_train)

# Prédictions
y_pred = multi_xgb.predict(X_test_scaled)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Résultats
print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)


📏 MAE : 0.4859257638454437
📏 MSE : 0.4079746901988983
📏 RMSE : 0.6387289645842736
📈 R²  : 0.9345407485961914


In [14]:
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Création du modèle MLP
mlp = MLPRegressor(hidden_layer_sizes=(100, 100), max_iter=1000, random_state=42)
multi_mlp = MultiOutputRegressor(mlp)

# Entraînement
multi_mlp.fit(X_train_scaled, y_train)

# Prédictions
y_pred = multi_mlp.predict(X_test_scaled)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Résultats
print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)



📏 MAE : 0.6908366114186995
📏 MSE : 0.8029085289494091
📏 RMSE : 0.8960516329706727
📈 R²  : 0.8711738988717546


Optimisation********************************

In [16]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. Créer le modèle de base
xgb = XGBRegressor(objective='reg:squarederror', random_state=42)

# 2. Définir la grille de paramètres
param_grid = {
    'n_estimators': [100, 200,300,500,1000],
    'max_depth': [3, 5, 7,8,10,12],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# 3. Grid Search avec validation croisée 5-fold
grid_search = GridSearchCV(estimator=xgb,
                           param_grid=param_grid,
                           cv=5,
                           scoring='r2',
                           verbose=1,
                           n_jobs=-1)

# 4. Entraîner
grid_search.fit(X_train, y_train)

# 5. Meilleurs paramètres
print("🔍 Best parameters found: ", grid_search.best_params_)

# 6. Meilleur modèle
best_model = grid_search.best_estimator_

# 7. Prédictions
y_pred = best_model.predict(X_test)

# 8. Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\n📏 MAE  : {mae}")
print(f"📏 MSE  : {mse}")
print(f"📏 RMSE : {rmse}")
print(f"📈 R²   : {r2}")


Fitting 5 folds for each of 360 candidates, totalling 1800 fits
🔍 Best parameters found:  {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 1000, 'subsample': 0.8}

📏 MAE  : 0.4145987629890442
📏 MSE  : 0.31107866764068604
📏 RMSE : 0.5577442672414357
📈 R²   : 0.9500876665115356


In [25]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 🔧 Meilleurs hyperparamètres trouvés
best_params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 1000,
    'subsample': 0.8
}

# 🚀 Entraîner le modèle avec ces paramètres
best_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    **best_params
)

best_model.fit(X_train, y_train)

# 🔍 Prédictions
y_pred = best_model.predict(X_test)

# 📊 Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\n📏 MAE  : {mae}")
print(f"📏 MSE  : {mse}")
print(f"📏 RMSE : {rmse}")
print(f"📈 R²   : {r2}")



📏 MAE  : 0.4145987629890442
📏 MSE  : 0.31107866764068604
📏 RMSE : 0.5577442672414357
📈 R²   : 0.9500876665115356


In [26]:
joblib.dump(best_model, "../SaveModels/xgboostTempwithoutscaling.pkl")
print("\n💾 Modèle sauvegardé sous 'xgboostTemp.pkl'.")



💾 Modèle sauvegardé sous 'xgboostTemp.pkl'.


In [18]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# ⚙️ Préparation des données
X = df.drop(columns=["Next_Tmin", "Next_Tmax"])  # Exemple de caractéristiques d'entrée
y = df[['Next_Tmin']] 

# ✂️ Split des données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🔁 Initialisation du modèle complet
multi_xgb = XGBRegressor(objective='reg:squarederror', random_state=42)
multi_xgb.fit(X_train, y_train)

# 🎯 Évaluation initiale
y_pred = multi_xgb.predict(X_test)
initial_mae = mean_absolute_error(y_test, y_pred)
initial_mse = mean_squared_error(y_test, y_pred)
initial_rmse = np.sqrt(initial_mse)
initial_r2 = r2_score(y_test, y_pred)

print(f"Initial performance: MAE={initial_mae}, MSE={initial_mse}, RMSE={initial_rmse}, R²={initial_r2}")

# 📊 Importance des features
importances = multi_xgb.feature_importances_
feature_ranking = pd.Series(importances, index=X.columns).sort_values()

# 🧠 Variables pour suivre le meilleur résultat
features_to_keep = list(feature_ranking.index)
best_r2 = initial_r2
best_features = features_to_keep.copy()
best_X_train = X_train.copy()
best_X_test = X_test.copy()
best_model = multi_xgb

# 🔁 Boucle de suppression des features les moins importantes
for i in range(len(feature_ranking)):
    feature_to_remove = feature_ranking.index[i]
    features_to_keep.remove(feature_to_remove)

    # Réduction du dataset
    X_train_sel = X_train[features_to_keep]
    X_test_sel = X_test[features_to_keep]

    # Réentraînement
    model_sel = XGBRegressor(objective='reg:squarederror', random_state=42)
    model_sel.fit(X_train_sel, y_train)

    # Prédictions
    y_pred_sel = model_sel.predict(X_test_sel)

    # Nouvelle performance
    r2 = r2_score(y_test, y_pred_sel)
    mae = mean_absolute_error(y_test, y_pred_sel)
    mse = mean_squared_error(y_test, y_pred_sel)
    rmse = np.sqrt(mse)

    print(f"\nAfter removing '{feature_to_remove}':")
    print(f"MAE={mae}, MSE={mse}, RMSE={rmse}, R²={r2}")

    # Comparaison avec la meilleure performance
    if r2 < best_r2:
        print(f"\nPerformance dropped, reverting to the previous optimal solution.")
        # Revenir à la meilleure configuration
        X_train = best_X_train
        X_test = best_X_test
        multi_xgb = best_model
        break
    else:
        # Sauvegarder la meilleure configuration
        best_r2 = r2
        best_features = features_to_keep.copy()
        best_X_train = X_train_sel.copy()
        best_X_test = X_test_sel.copy()
        best_model = model_sel

# 🏁 Résultat final
print("\n✅ Final selected features:")
print(best_features)


Initial performance: MAE=0.44981175661087036, MSE=0.36509838700294495, RMSE=0.6042337188563255, R²=0.9414201974868774

After removing 'month':
MAE=0.45154139399528503, MSE=0.3690008521080017, RMSE=0.6074544033160034, R²=0.940794050693512

Performance dropped, reverting to the previous optimal solution.

✅ Final selected features:
['month', 'LDAPS_LH', 'LDAPS_Tmax_lapse', 'LDAPS_RHmax', 'station', 'LDAPS_WS', 'LDAPS_CC3', 'LDAPS_CC2', 'LDAPS_CC1', 'Solar radiation', 'LDAPS_RHmin', 'Present_Tmax', 'LDAPS_PPT4', 'LDAPS_PPT3', 'LDAPS_PPT2', 'LDAPS_CC4', 'day', 'LDAPS_PPT1', 'lat', 'year', 'lon', 'Slope', 'DEM', 'Present_Tmin', 'LDAPS_Tmin_lapse']


In [37]:
# 📌 Importer les bibliothèques nécessaires
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef


# 📌 Appliquer PCA
pca = PCA(n_components=0.999999999)  # Garder 95% de la variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"🔍 Nombre de features après PCA : {X_train_pca.shape[1]}")


🔍 Nombre de features après PCA : 25


In [34]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 🔧 Meilleurs hyperparamètres trouvés
best_params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 1000,
    'subsample': 0.8
}

# 🚀 Entraîner le modèle avec ces paramètres
best_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    **best_params
)

best_model.fit(X_train_pca, y_train)

# 🔍 Prédictions
y_pred = best_model.predict(X_test_pca)

# 📊 Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\n📏 MAE  : {mae}")
print(f"📏 MSE  : {mse}")
print(f"📏 RMSE : {rmse}")
print(f"📈 R²   : {r2}")



📏 MAE  : 0.47408217191696167
📏 MSE  : 0.39458999037742615
📏 RMSE : 0.6281639836678208
📈 R²   : 0.9366883039474487


optimisation RandomForest *****************************************

In [39]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# ⚙️ Préparation des données
X = df.drop(columns=["Next_Tmin", "Next_Tmax"])  # Exemple de caractéristiques d'entrée
y = df[['Next_Tmin']]

# ✂️ Split des données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🔁 Initialisation du modèle complet
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train.values.ravel())  # .ravel() car RandomForest attend un vecteur pour y

# 🎯 Évaluation initiale
y_pred = rf.predict(X_test)
initial_mae = mean_absolute_error(y_test, y_pred)
initial_mse = mean_squared_error(y_test, y_pred)
initial_rmse = np.sqrt(initial_mse)
initial_r2 = r2_score(y_test, y_pred)

print(f"Initial performance: MAE={initial_mae}, MSE={initial_mse}, RMSE={initial_rmse}, R²={initial_r2}")

# 📊 Importance des features
importances = rf.feature_importances_
feature_ranking = pd.Series(importances, index=X.columns).sort_values()

# 🧠 Variables pour suivre le meilleur résultat
features_to_keep = list(feature_ranking.index)
best_r2 = initial_r2
best_features = features_to_keep.copy()
best_X_train = X_train.copy()
best_X_test = X_test.copy()
best_model = rf

# 🔁 Boucle de suppression des features les moins importantes
for i in range(len(feature_ranking)):
    feature_to_remove = feature_ranking.index[i]
    features_to_keep.remove(feature_to_remove)

    # Réduction du dataset
    X_train_sel = X_train[features_to_keep]
    X_test_sel = X_test[features_to_keep]

    # Réentraînement
    model_sel = RandomForestRegressor(n_estimators=100, random_state=42)
    model_sel.fit(X_train_sel, y_train.values.ravel())

    # Prédictions
    y_pred_sel = model_sel.predict(X_test_sel)

    # Nouvelle performance
    r2 = r2_score(y_test, y_pred_sel)
    mae = mean_absolute_error(y_test, y_pred_sel)
    mse = mean_squared_error(y_test, y_pred_sel)
    rmse = np.sqrt(mse)

    print(f"\nAfter removing '{feature_to_remove}':")
    print(f"MAE={mae}, MSE={mse}, RMSE={rmse}, R²={r2}")

    # Comparaison avec la meilleure performance
    if r2 < best_r2:
        print(f"\n⚠️ Performance dropped, reverting to the previous optimal solution.")
        # Revenir à la meilleure configuration
        X_train = best_X_train
        X_test = best_X_test
        rf = best_model
        break
    else:
        # Sauvegarder la meilleure configuration
        best_r2 = r2
        best_features = features_to_keep.copy()
        best_X_train = X_train_sel.copy()
        best_X_test = X_test_sel.copy()
        best_model = model_sel

# 🏁 Résultat final
print("\n✅ Final selected features:")
print(best_features)


Initial performance: MAE=0.5623249516441008, MSE=0.5618280851063834, RMSE=0.7495519228888572, R²=0.909855084235674

After removing 'month':
MAE=0.5624551901998711, MSE=0.5619757872340427, RMSE=0.7496504433628001, R²=0.9098313855345783

⚠️ Performance dropped, reverting to the previous optimal solution.

✅ Final selected features:
['month', 'LDAPS_PPT3', 'LDAPS_PPT4', 'LDAPS_PPT2', 'year', 'LDAPS_PPT1', 'lat', 'day', 'Slope', 'station', 'lon', 'LDAPS_LH', 'DEM', 'LDAPS_CC2', 'LDAPS_CC3', 'LDAPS_CC1', 'Solar radiation', 'LDAPS_CC4', 'LDAPS_RHmin', 'LDAPS_RHmax', 'LDAPS_Tmax_lapse', 'Present_Tmax', 'LDAPS_WS', 'Present_Tmin', 'LDAPS_Tmin_lapse']


For Preventing max temperature

In [47]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Définir les caractéristiques (X) et les cibles (y)
X = df.drop(columns=["Next_Tmin", "Next_Tmax"])  # Exemple de caractéristiques d'entrée
y = df[['Next_Tmax']]  # Variables cibles : température minimale et maximale futures

# Séparer les données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Appliquer le MinMaxScaler
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [48]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
# Créer et entraîner le modèle de régression linéaire
model = LinearRegression()
model.fit(X_train_scaled, y_train)
# Faire des prédictions
y_pred = model.predict(X_test_scaled)

# MAE : Erreur absolue moyenne
mae = mean_absolute_error(y_test, y_pred)

# MSE : Erreur quadratique moyenne
mse = mean_squared_error(y_test, y_pred)

# RMSE : Racine de l’erreur quadratique moyenne
rmse = np.sqrt(mse)

# R² : Coefficient de détermination
r2 = r2_score(y_test, y_pred)

# Affichage
print("📏 MAE  :", mae)
print("📏 MSE  :", mse)
print("📏 RMSE :", rmse)
print("📈 R²   :", r2)


📏 MAE  : 1.207114940461402
📏 MSE  : 2.6516193525994582
📏 RMSE : 1.628379363845986
📈 R²   : 0.7285746031648188


In [49]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor

# Création du modèle multi-sortie
rf = RandomForestRegressor(n_estimators=100, random_state=42)
multi_rf = MultiOutputRegressor(rf)

# Entraînement
multi_rf.fit(X_train_scaled, y_train)

# Prédictions
y_pred = multi_rf.predict(X_test_scaled)

# Évaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)


📏 MAE : 0.690275306254029
📏 MSE : 0.8293038729851702
📏 RMSE : 0.910661228440725
📈 R²  : 0.9151106916604428


In [50]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 🔧 Meilleurs hyperparamètres trouvés
best_params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 1000,
    'subsample': 0.8
}

# 🚀 Entraîner le modèle avec ces paramètres
best_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    **best_params
)

best_model.fit(X_train, y_train)

# 🔍 Prédictions
y_pred = best_model.predict(X_test)

# 📊 Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\n📏 MAE  : {mae}")
print(f"📏 MSE  : {mse}")
print(f"📏 RMSE : {rmse}")
print(f"📈 R²   : {r2}")



📏 MAE  : 0.534692108631134
📏 MSE  : 0.4943082928657532
📏 RMSE : 0.7030706172681043
📈 R²   : 0.9494015574455261


In [51]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 🔧 Meilleurs hyperparamètres trouvés
best_params = {
    'colsample_bytree': 0.8,
    'learning_rate': 0.1,
    'max_depth': 5,
    'n_estimators': 1000,
    'subsample': 0.8
}

# 🚀 Entraîner le modèle avec ces paramètres
best_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    **best_params
)

best_model.fit(X_train_scaled, y_train)

# 🔍 Prédictions
y_pred = best_model.predict(X_test_scaled)

# 📊 Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\n📏 MAE  : {mae}")
print(f"📏 MSE  : {mse}")
print(f"📏 RMSE : {rmse}")
print(f"📈 R²   : {r2}")



📏 MAE  : 0.5298702120780945
📏 MSE  : 0.4864727854728699
📏 RMSE : 0.6974760106791271
📈 R²   : 0.9502035975456238


In [52]:
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Création du modèle MLP
mlp = MLPRegressor(hidden_layer_sizes=(100, 100), max_iter=1000, random_state=42)
multi_mlp = MultiOutputRegressor(mlp)

# Entraînement
multi_mlp.fit(X_train_scaled, y_train)

# Prédictions
y_pred = multi_mlp.predict(X_test_scaled)

# Évaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

# Résultats
print("📏 MAE :", mae)
print("📏 MSE :", mse)
print("📏 RMSE :", rmse)
print("📈 R²  :", r2)



📏 MAE : 0.8901343648883997
📏 MSE : 1.3337637946370136
📏 RMSE : 1.15488691854961
📈 R²  : 0.863473101111269


In [53]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 🧪 Tu peux tester différents nombres de voisins (n_neighbors)
knn_model = KNeighborsRegressor(n_neighbors=5)

# 🚀 Entraînement
knn_model.fit(X_train_scaled, y_train)

# 🔍 Prédictions
y_pred_knn = knn_model.predict(X_test_scaled)

# 📊 Évaluation
mae_knn = mean_absolute_error(y_test, y_pred_knn)
mse_knn = mean_squared_error(y_test, y_pred_knn)
rmse_knn = np.sqrt(mse_knn)
r2_knn = r2_score(y_test, y_pred_knn)

print(f"\n📏 MAE  : {mae_knn}")
print(f"📏 MSE  : {mse_knn}")
print(f"📏 RMSE : {rmse_knn}")
print(f"📈 R²   : {r2_knn}")



📏 MAE  : 0.9600644745325597
📏 MSE  : 1.6552059316569956
📏 RMSE : 1.2865480681486392
📈 R²   : 0.8305696002695415


In [54]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 💡 Initialisation du modèle SVR avec des paramètres de base
svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)

# 🚀 Entraînement
svr_model.fit(X_train_scaled, y_train)

# 🔍 Prédictions
y_pred_svr = svr_model.predict(X_test_scaled)

# 📊 Évaluation
mae_svr = mean_absolute_error(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mse_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print(f"\n📏 MAE  : {mae_svr}")
print(f"📏 MSE  : {mse_svr}")
print(f"📏 RMSE : {rmse_svr}")
print(f"📈 R²   : {r2_svr}")


c:\Users\T U F\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



📏 MAE  : 0.8886660781002134
📏 MSE  : 1.445457152554252
📏 RMSE : 1.2022716633749013
📈 R²   : 0.8520399314269325


In [55]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 🌱 Initialisation du modèle
tree_model = DecisionTreeRegressor(random_state=42)

# 🚀 Entraînement
tree_model.fit(X_train_scaled, y_train)

# 🔍 Prédictions
y_pred_tree = tree_model.predict(X_test_scaled)

# 📊 Évaluation
mae_tree = mean_absolute_error(y_test, y_pred_tree)
mse_tree = mean_squared_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mse_tree)
r2_tree = r2_score(y_test, y_pred_tree)

print(f"\n📏 MAE  : {mae_tree}")
print(f"📏 MSE  : {mse_tree}")
print(f"📏 RMSE : {rmse_tree}")
print(f"📈 R²   : {r2_tree}")



📏 MAE  : 1.0068343004513218
📏 MSE  : 1.9748549323017408
📏 RMSE : 1.4052953185369048
📈 R²   : 0.7978496486811225


In [56]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 🌳 Initialisation du modèle
extra_trees_model = ExtraTreesRegressor(n_estimators=100, random_state=42)

# 🚀 Entraînement
extra_trees_model.fit(X_train_scaled, y_train)

# 🔍 Prédictions
y_pred_extra = extra_trees_model.predict(X_test_scaled)

# 📊 Évaluation
mae_extra = mean_absolute_error(y_test, y_pred_extra)
mse_extra = mean_squared_error(y_test, y_pred_extra)
rmse_extra = np.sqrt(mse_extra)
r2_extra = r2_score(y_test, y_pred_extra)

print(f"\n📏 MAE  : {mae_extra}")
print(f"📏 MSE  : {mse_extra}")
print(f"📏 RMSE : {rmse_extra}")
print(f"📈 R²   : {r2_extra}")


c:\Users\T U F\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



📏 MAE  : 0.5840309477756286
📏 MSE  : 0.599186578981302
📏 RMSE : 0.7740714301544154
📈 R²   : 0.938665987326243


In [58]:
joblib.dump(best_model, "../SaveModels/xgboost_modelMaxTemp.pkl")
print("\n💾 Modèle sauvegardé sous 'xgboost_model2v1.pkl'.")



💾 Modèle sauvegardé sous 'xgboost_model2v1.pkl'.
